
Notebook for the initial testing of features

In [3]:
import httpx
import csv

### HTTP Data Extraction

In [4]:
url = f"https://repodatos.atdt.gob.mx/api_update/conanp/areas_destinadas_voluntariamente_conservacion/relacion_advc.csv"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/156.0.0.0 Safari/537.36",
    "Referer": "https://www.datos.gob.mx/",
    "Accept": "text/csv,*/*;q=0.8",
    "Accept-Language": "es-MX,es;q=0.9,en;q=0.8",
}

In [9]:
with httpx.Client(headers=headers, follow_redirects=True, timeout=60, http2= True) as client:
    with client.stream("GET", url) as r:
        r.read()   # ← required
        print(r.status_code)
        data = r.json()['result']['records']
        print(data)
        fieldnames = list(set().union(*data))
        with open(f"d.csv", "wt", newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames, restval='') 
            writer.writeheader()
            writer.writerows(data)


200


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

### Data processing

In [10]:
import os

In [27]:
"_".join('notas_impacto_ambiental_resolved_20261119162428'.split('_')[0:-1])

'notas_impacto_ambiental_resolved'

In [23]:
files = os.listdir('../grdata/_raw')

for file in files:
    print(file)
    if file in ['advc_20261118145643.csv','areas_protegidas_20261118151600.csv']:
        os.rename('../grdata/_raw' + '/' + file, '../grdata/_processed' + '/' + file)
    else:
        continue

advc_20261118145643.csv
advc_20261118151600.csv
advc_20261118152946.csv
advc_20261118153122.csv
advc_20261119161353.csv
advc_20261119161426.csv
advc_20261119161503.csv
areas_protegidas_20261118151600.csv
areas_protegidas_20261118152946.csv
areas_protegidas_20261119161503.csv
biosferas_UNESCO_20261118151600.csv
biosferas_UNESCO_20261118152946.csv
biosferas_UNESCO_20261119161503.csv
generacion_res_pel_tipo_20261118151600.csv
generacion_res_pel_tipo_20261118152946.csv
generacion_res_pel_tipo_20261119161503.csv
historico_incendios_forestales_20261118151600.csv
historico_incendios_forestales_20261118152946.csv
historico_incendios_forestales_20261119161503.csv
incendios_forestales_20261118151600.csv
incendios_forestales_20261118152946.csv
incendios_forestales_20261119161503.csv
notas_impacto_ambiental_20261118152946.csv
notas_impacto_ambiental_20261119161503.csv
notas_impacto_ambiental_resolved_20261118152946.csv
notas_impacto_ambiental_resolved_20261119161503.csv
patrimonios_20261118151600.

### Table readings

In [1]:
import polars as pl

In [41]:
tables = os.listdir('../grdata/_processed')

for i, table in enumerate(tables, 1):
    print(i, table)
    #print(pl.read_parquet('../grdata/_processed/' + table))
    print("""************************************************************""")


1 advc.parquet
************************************************************
2 areas_protegidas.parquet
************************************************************
3 biosferas_UNESCO.parquet
************************************************************
4 generacion_res_pel_tipo.parquet
************************************************************
5 historico_incendios_forestales.parquet
************************************************************
6 incendios_forestales.parquet
************************************************************
7 notas_impacto_ambiental.parquet
************************************************************
8 notas_impacto_ambiental_resolved.parquet
************************************************************
9 patrimonios.parquet
************************************************************
10 sitios_cont.parquet
************************************************************
11 sitios_cont_tipo.parquet
************************************************************
12 s

### Protected Zones Transformation

In [2]:
import polars as pl
from IPython.display import Markdown, display, HTML


l = ['advc.parquet', 'areas_protegidas.parquet', 'biosferas_UNESCO.parquet', 'patrimonios.parquet']
PATH = '../grdata/_processed/'

for file in l:
    table = pl.read_parquet(PATH + file)
    print(file)
    from IPython.display import Markdown, display

    html = '<div style="display:flex; gap:24px;">'
    html += f'<div>{table._repr_html_()}</div>'
    html += '</div>'
    display(HTML(html))


advc.parquet


numero,numero_certificado,nombre_area,superficie_certificada_ha,municipio,estado,anio_certificacion,vigencia_anio,tipo_propietario,region_conanp
i64,str,str,f64,str,str,i64,f64,str,str
1,"""CONANP-01/2002""","""Parque Ecológico Jaguaroundi""",960.734884,"""Coatzacoalcos""","""Veracruz""",2002,99.0,"""Empresa de Participación Estat…","""Planicie Costera y Golfo de Mé…"
2,"""CONANP-02/2003""","""Rancho La Asunción con el Area…",572.0,"""Tacotalpa""","""Tabasco""",2003,null,"""Persona física""","""Planicie Costera y Golfo de Mé…"
3,"""CONANP-10/2004""","""Cerro Chango""",700.0,"""Santiago Jocotepec""","""Oaxaca""",2004,null,"""Ejido (Tierras de Uso Común)""","""Frontera Sur Istmo y Pacífico …"
4,"""CONANP-13/2004""","""Área de Conservación San Pedro…",5050.0,"""San Felipe Usila""","""Oaxaca""",2004,25.0,"""Comunidad (Bienes comunales)""","""Frontera Sur Istmo y Pacífico …"
5,"""CONANP-14/2004""","""Área de Conservación Santiago …",4300.0,"""San Felipe Usila""","""Oaxaca""",2004,25.0,"""Comunidad (Bienes comunales)""","""Frontera Sur Istmo y Pacífico …"
…,…,…,…,…,…,…,…,…,…
615,"""CONANP-649/2025""","""CEMEX Planta Valles""",151.29095,"""Ciudad Valles""","""San Luis Potosí""",2025,15.0,"""Persona moral""","""Noreste y Sierra Madre Orienta…"
616,"""CONANP-650/2026""","""Samachique""",4921.0,"""Guachochi""","""Chihuahua""",2026,99.0,"""Ejido (Tierras de Uso Común)""","""Norte y Sierra Madre Occidenta…"
617,"""CONANP-651/2026""","""Boyte´Tik""",500.0,"""San Cristóbal de las Casas""","""Chiapas""",2026,15.0,"""Ejido (Tierras de Uso Común)""","""Frontera Sur Istmo y Pacífico …"


areas_protegidas.parquet


sup_ha,fecha_dec,fecha_pm,num_anp,ficha_simec,region,cat_manejo,sterr_ha,nom_direc,cert_sinap,cui_dof,fecha_otro,smar_ha,estado,cui_anp,_id,nombre,desig_inter
f64,str,str,f64,str,str,str,f64,str,str,str,str,f64,str,str,i64,str,str
934756.25,"""1993-06-10T00:00:00""","""2009-09-25T00:00:00""",110.0,"""https://simec.conanp.gob.mx/fi…","""Noroeste y Alto Golfo de Calif…","""RB""",407147.5456,"""M. en A. Domingo de Jesús Zata…","""SINAP 004""","""2.1.01.110""","""1993-06-10T00:00:00""",527608.7044,"""Baja California y Sonora""","""ANP00201110""",1,"""Alto Golfo de California y Del…","""World Heritage - Patrimonio Mu…"
333768.505,"""1994-06-06T00:00:00""","""2007-11-29T00:00:00""",117.0,"""https://simec.conanp.gob.mx/fi…","""Península de Yucatán y Caribe …","""PN""",53.001756,"""Biól. Cristóbal Enrique Cácere…","""SINAP 049""","""9.3.03.117""","""2000-06-07T00:00:00""",333715.5032,"""Yucatán""","""ANP00903117""",2,"""Arrecife Alacranes""","""Sitio RAMSAR - Convención de H…"
9066.6311,"""1998-02-02T00:00:00""","""2000-09-18T00:00:00""",133.0,"""https://simec.conanp.gob.mx/fi…","""Península de Yucatán y Caribe …","""PN""",37.743644,"""Biól. Juan Carlos Romero Gil""","""SINAP 044""","""9.3.07.133""","""1998-02-02T00:00:00""",9028.887456,"""Quintana Roo""","""ANP00903133""",3,"""Arrecife de Puerto Morelos""","""World Heritage - Patrimonio Mu…"
11987.875,"""1996-07-19T00:00:00""","""1998-10-02T00:00:00""",128.0,"""https://simec.conanp.gob.mx/fi…","""Península de Yucatán y Caribe …","""PN""",82.278916,"""Biól. Blanca Alicia Quiroga Ga…","""SINAP 046""","""9.3.05.128""","""2000-06-07T00:00:00""",11905.59608,"""Quintana Roo""","""ANP00903128""",4,"""Arrecifes de Cozumel""","""World Heritage - Patrimonio Mu…"
34927.1584,"""1998-02-02T00:00:00""","""2011-06-13T00:00:00""",132.0,"""https://simec.conanp.gob.mx/fi…","""Península de Yucatán y Caribe …","""RB""",1361.001633,"""M. en C. Oscar Guzmán Escalant…","""SINAP 015""","""9.1.04.132""","""1998-02-02T00:00:00""",33566.15677,"""Quintana Roo""","""ANP00901132""",5,"""Arrecifes de Sian Ka'an""","""Sitio RAMSAR - Convención de H…"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
641284.7374,"""2000-11-27T00:00:00""","""2022-08-05T00:00:00""",147.0,"""https://simec.conanp.gob.mx/fi…","""Occidente y Pacífico Centro""","""RB""",24295.16619,"""Biól. Marisol Amador Medina""",null,"""5.1.03.147""","""2000-11-27T00:00:00""",616989.5712,"""Nayarit""","""ANP00501147""",96,"""Islas Marías""","""MaB - Hombre y Biosfera"""
1383.019695,"""2005-04-25T00:00:00""","""2011-02-25T00:00:00""",159.0,"""https://simec.conanp.gob.mx/fi…","""Occidente y Pacífico Centro""","""PN""",71.163268,"""M. en C. Areli Zuleth Cueto Va…",null,"""5.3.08.159""","""2005-04-25T00:00:00""",1311.856427,"""Nayarit""","""ANP00503159""",97,"""Islas Marietas""","""Sitio RAMSAR - Convención de H…"
39819.086,"""1935-11-08T00:00:00""","""2013-04-02T00:00:00""",2.0,"""https://simec.conanp.gob.mx/fi…","""Centro y Eje Neovolcánico""","""PN""",39819.086,"""Biól. Francisco Javier Salazar…",null,"""6.3.02.002""","""1948-02-11T00:00:00""",0.0,"""Estado de México, Puebla y Mor…","""ANP00603002""",98,"""Iztaccíhuatl-Popocatépetl""","""MaB - Hombre y Biosfera"""


biosferas_UNESCO.parquet


estado_que_abarca,numero,region,reserva_biosfera_mab,nombre_director,anio_designacion,superficie_total_ha,anp_involucrada_rbmab,_id,anio_modificacion_superficie
str,i64,str,str,str,i64,f64,str,i64,str
"""Coahuila-Durango-Chihuahua""",1,"""Noreste y Sierra Madre Orienta…","""Mapimi""","""Juan Carlos Ibarra Flores""",1977,342387.0,"""Mapimi""",1,"""sin dato"""
"""Durango""",2,"""Norte y Sierra Made Occidental""","""La Michilia""","""Candelario Cardenas Figeroa""",1977,35932.5,"""La Michilia""",2,"""sin dato"""
"""Chiapas""",3,"""Frontera Sur - Istmo y Pacific…","""Montes Azules""","""Jose Nava Hernandez""",1979,331200.0,"""Montes Azules""",3,"""sin dato"""
"""Tamaulipas""",4,"""Planicie Costera y Golfo de Me…","""El Cielo""","""Daniel Alejandro Penaloza Mede…",1986,144530.0,"""El Cielo""",4,"""sin dato"""
"""Quintana Roo""",5,"""Península de Yucatán y Caribe …","""Sian Ka an""","""Oscar Guzman Escalante""",1986,528147.66,"""Sian Ka an""",5,"""sin dato"""
…,…,…,…,…,…,…,…,…,…
"""Chiapas""",37,"""Frontera Sur - Istmo y Pacific…","""Naha Metzabok""","""Pavel Palacios Chavez""",2010,43362.0,"""Naha-Metzabok""",37,"""sin dato"""
"""Estado de Mexico-Puebla-Morelo…",38,"""Centro y Eje Neovolcánico""","""Los Volcanes""","""Marco Antonio Castro Martinez""",2010,171774.41,"""Iztacihuatl Popocatepetl""",38,"""sin dato"""
"""Nayarit""",39,"""Occidente y Pacífico Centro""","""Islas Marias""","""Pablo Zamorano de Haro""",2010,641284.73,"""Islas Marias""",39,"""sin dato"""


patrimonios.parquet


numero,categoria_manejo_anp,lista_patrimonio_mundial_peligro,nombre_director,tipo_sitio,fecha_inscripcion,estado,criterios_inscripcion,_id,sitio_patrimonio_mundial,anp_involucrada_sitio,superficie_total_(ha),url_patrimonio_mundial
i64,str,str,str,str,i64,str,str,i64,str,str,f64,str
1,"""Reserva de Biosfera""","""No""","""Oscar Guzmán Escalante""","""Natural""",1987,"""Quintana Roo""","""VII-X""",1,"""Sian Kaan""","""Sian Kaan""",528000.0,"""https://whc.unesco.org/en/list…"
2,"""Reserva de Biosfera""","""No""","""Marco Antonio González Vizcarr…","""Natural""",1993,"""Baja California Sur""","""X""",2,"""Santuario de Ballenas del Vizc…","""El Vizcaino""",369631.0,"""https://whc.unesco.org/en/list…"
3,"""Reserva de Biosfera""","""SI""","""Jesús Zatarain González""","""Natural""",2005,"""Baja California""","""VII-IX-X""",3,"""Islas y Areas Protegidas del G…","""Alto Golfo de California y Del…",688558.0,"""https://whc.unesco.org/en/list…"
3,"""Area de Protección de Flora y …","""SI""","""Ivette Guadalupe Astorga Monto…","""Natural""",2005,"""Baja California""","""VII-IX-X""",4,"""Islas y Areas Protegidas del G…","""Islas del Golfo de California""",688558.0,"""https://whc.unesco.org/en/list…"
3,"""Area de Protección de Flora y …","""SI""","""Ivette Guadalupe Astorga Monto…","""Natural""",2005,"""Baja California Sur""","""VII-IX-X""",5,"""Islas y Areas Protegidas del G…","""Islas del Golfo de California""",688558.0,"""https://whc.unesco.org/en/list…"
…,…,…,…,…,…,…,…,…,…,…,…,…
5,"""Reserva de Biosfera""","""No""","""Martín Sau Cota,""","""Natural""",2013,"""Sonora""","""VII-VIII-X""",20,"""El Pinacate y Gran Desierto de…","""El Pinacate y Gran Desierto de…",714556.0,"""https://whc.unesco.org/en/list…"
6,"""Reserva de Biosfera""","""No""","""José Adalberto Zuiñiga Morales""","""Mixto""",2014,"""Campeche""","""I-II-III-IV-IX-X""",21,"""Antigua Ciudad Maya y Bosques …","""Calakmul""",331397.0,"""https://whc.unesco.org/en/list…"
7,"""Parque Nacional""","""No""","""Luz Eréndida Frías Hernández""","""Natural""",2016,"""Oceano Pacífico""","""VII-IX-X""",22,"""Archipielago de Revillagigedo""","""Revillagigedo""",636685.375,"""https://whc.unesco.org/en/list…"


In [2]:
pl.read_parquet(PATH + 'advc.parquet').columns

['numero',
 'numero_certificado',
 'nombre_area',
 'superficie_certificada_ha',
 'municipio',
 'estado',
 'anio_certificacion',
 'vigencia_anio',
 'tipo_propietario',
 'region_conanp']

##### Dictionary of variables:
| Column | advc | areas_protegidas | biosferas_UNESCO | patrimonios |
| -------- | -------- | -------- | -------- | -------- |
| municipality | municipio | Null | Null | Null |
| state | estado | estado | estado_que_abarca |estado|
| place_name | nombre_area | nombre | reserva_biosfera_mab | sitio_patrimonio_mundial |
| owner_type | tipo_propietario | desig_inter | Null | Null | 
| certification | numero_certificado | cert_sinap o cui_anp | Null | Null |
| owner | tipo_propietario | nom_direc | nombre_director | nombre_director |
| surface | superficie_certificada_ha | sup_ha | superficie_total_ha | superficie_total_(ha) |
| is_endangered? | Null | Null | Null | Null |  


In [8]:
# import json

# def change_col_names(df: pl.DataFrame, old_name: str, new_name: str) -> pl.DataFrame:
#     if old_name not in df.columns:
#         return df.with_columns(pl.lit(None).alias(new_name))
#     else:
#         return df.rename({old_name: new_name}, strict=False)

with open("../src/data_mapping.json") as mappings_file:
    mappings = json.load(mappings_file)

frames = {name: pl.read_parquet(PATH + name + '.parquet') for name in mappings}
# print(frames)

for name in mappings:
    df = frames[name][0]
    for old, new in mappings[name].items():
        # print(name)
        # print(type(old))
        # print(type(frames[name][0]))
        df = change_col_names(df = df, old_name = old, new_name = new)
    # print(df.columns)
    f = df.select( ["municipality",
        "state",
        "place_name",
        "owner_type",
        "certification",
        "owner",
        "is_endangered?"]
    )
    print(f)
     
    


shape: (1, 7)
┌───────────────┬──────────┬───────────────┬───────────────┬───────────────┬───────┬───────────────┐
│ municipality  ┆ state    ┆ place_name    ┆ owner_type    ┆ certification ┆ owner ┆ is_endangered │
│ ---           ┆ ---      ┆ ---           ┆ ---           ┆ ---           ┆ ---   ┆ ?             │
│ str           ┆ str      ┆ str           ┆ str           ┆ str           ┆ null  ┆ ---           │
│               ┆          ┆               ┆               ┆               ┆       ┆ null          │
╞═══════════════╪══════════╪═══════════════╪═══════════════╪═══════════════╪═══════╪═══════════════╡
│ Coatzacoalcos ┆ Veracruz ┆ Parque        ┆ Empresa de    ┆ CONANP-01/200 ┆ null  ┆ null          │
│               ┆          ┆ Ecológico     ┆ Participación ┆ 2             ┆       ┆               │
│               ┆          ┆ Jaguaroundi   ┆ Estat…        ┆               ┆       ┆               │
└───────────────┴──────────┴───────────────┴───────────────┴───────────────┴─

In [6]:
import json
def change_col_names(df: pl.DataFrame, old_name: str, new_name: str) -> pl.DataFrame:

    if old_name in df.columns:
        return df.rename({old_name: new_name}, strict=False)
    else:
        return df.with_columns(pl.lit(None).alias(new_name))
    
def protected_zones_stantization(mapping: str, mappings: dict[str, dict], dataframes: dict[str, pl.DataFrame]):

    for old, new in mappings[mapping].items():
        dataframes[mapping] = change_col_names(df = dataframes[mapping], old_name = old, new_name = new)

    print(dataframes[mapping].columns)
    std_df = dataframes[mapping].select( ["municipality",
        "state",
        "place_name",
        "owner_type",
        "certification",
        "owner",
        "surface",
        "is_endangered?"]
    )
    return std_df


def protected_zones_concat(path: str,json_path: str = "../src/data_mapping.json"):
    # receive the mappings
    with open(json_path) as mappings_file:
        mappings = json.load(mappings_file)

    frames = {name: pl.read_parquet(path + name + '.parquet') for name in mappings}

    protected_zones_df = pl.DataFrame(schema={
        "municipality": pl.Utf8,
        "state": pl.Utf8,
        "place_name": pl.Utf8,
        "owner_type": pl.Utf8,
        "certification": pl.Utf8,
        "owner": pl.Utf8,
        "surface": pl.Float64,
        "is_endangered?": pl.Boolean
    })    
    # display(frames)
    for frame in frames:

        protected_zones_df = protected_zones_stantization(mappings = mappings,mapping = frame, dataframes= frames)
        print(protected_zones_df.shape[0], protected_zones_df.columns)
        # print(protected_zones_df)


protected_zones_concat(path=PATH)

['numero', 'certification', 'place_name', 'surface', 'municipality', 'state', 'anio_certificacion', 'vigencia_anio', 'owner_type', 'region_conanp', 'owner', 'is_endangered?']
619 ['municipality', 'state', 'place_name', 'owner_type', 'certification', 'owner', 'surface', 'is_endangered?']
['surface', 'fecha_dec', 'fecha_pm', 'num_anp', 'ficha_simec', 'region', 'cat_manejo', 'sterr_ha', 'owner', 'cert_sinap', 'cui_dof', 'fecha_otro', 'smar_ha', 'state', 'certification', '_id', 'place_name', 'owner_type', 'municipality', 'is_endangered?']
100 ['municipality', 'state', 'place_name', 'owner_type', 'certification', 'owner', 'surface', 'is_endangered?']
['state', 'numero', 'region', 'place_name', 'owner', 'anio_designacion', 'surface', 'anp_involucrada_rbmab', '_id', 'anio_modificacion_superficie', 'municipality', 'owner_type', 'certification', 'is_endangered?']
41 ['municipality', 'state', 'place_name', 'owner_type', 'certification', 'owner', 'surface', 'is_endangered?']
['numero', 'categoria